# Transfer Learning USAD → SIATA 203 (UNAN) — v.K
**Período:** 2023-01-01 → 2023-06-30  
**Objetivo:** Detectar anomalías de temperatura en el sensor 203 transfiriendo el modelo USAD pre-entrenado en SWaT (51 sensores) a 1 sensor mediante una submatriz adaptadora.

**Mejoras v.K sobre v.J:**
- Limpieza de datos: valores centinela (-999/-1000°C) → NaN → dropna
- Detección y corrección automática de score invertido (AUC < 0.5)
- Umbral F1-optimal en lugar de Youden J sobre ROC degradada
- Gradient clipping (max_norm=1.0) para estabilizar entrenamiento
- Gráficas de Error de Reconstrucción interactivas (Plotly)
- Curva Precision-Recall y histograma de scores por clase (Plotly)
- Gráficas de reconstrucción con eje de fechas (Plotly)

In [ ]:
# ── Celda 0: Clonar repo y configurar paths ──────────────────────────────────
import os, sys

REPO_URL = "https://github.com/ronvas234/data-science-monograph.git"
REPO_DIR = "data-science-monograph"
BRANCH   = 'feature/transfer-learning-plan-j'

os.system('apt-get install -y git-lfs 2>/dev/null')

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b {BRANCH} {REPO_URL}')

if os.path.basename(os.getcwd()) != REPO_DIR:
    os.chdir(REPO_DIR)

os.system('git lfs pull')

USAD_PATH = os.path.abspath("modelos/usad")
if USAD_PATH not in sys.path:
    sys.path.insert(0, USAD_PATH)

print(f"CWD: {os.getcwd()}")
print(f"USAD path: {USAD_PATH}")

In [ ]:
# ── Celda 1: Imports ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset
from dataclasses import dataclass
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    roc_curve, roc_auc_score, balanced_accuracy_score,
    confusion_matrix, classification_report, f1_score,
    precision_score, recall_score,
    precision_recall_curve, average_precision_score
)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from utils import get_default_device, to_device

device = get_default_device()
print(f"Device: {device}")

In [ ]:
# ── Celda 2: Configuración ────────────────────────────────────────────────────
@dataclass
class Config:
    station_code: int    = 203
    fecha_inicio: str    = "2023-01-01"
    fecha_fin: str       = "2023-06-30"
    data_path: str       = "modelos/usad/data/temperatura_estaciones_2020_2025.csv"
    model_path: str      = "modelos/usad/model.pth"
    window_size: int     = 12
    z_size: int          = 120
    train_frac: float    = 0.65
    val_frac: float      = 0.20
    epochs_frozen: int   = 20
    epochs_total: int    = 100
    batch_size: int      = 128
    lr_frozen: float     = 1e-3
    lr_finetune: float   = 1e-4
    weight_decay: float  = 1e-5
    alpha: float         = 0.5
    beta: float          = 0.5
    # Rango físico plausible de temperatura (Medellín)
    temp_min: float      = 5.0
    temp_max: float      = 45.0

cfg = Config()
print(cfg)

In [ ]:
# ── Celda 3: Carga y EDA ──────────────────────────────────────────────────────
df_raw = pd.read_csv(
    cfg.data_path,
    usecols=["codigo", "fecha_hora", "t", "calidad_dudosa"],
    parse_dates=["fecha_hora"],
    dtype={"codigo": int, "calidad_dudosa": str}
)

df = df_raw[
    (df_raw["codigo"] == cfg.station_code) &
    (df_raw["fecha_hora"] >= cfg.fecha_inicio) &
    (df_raw["fecha_hora"] <= cfg.fecha_fin)
].copy()

df = df.sort_values("fecha_hora").reset_index(drop=True)

df["label"] = (df["calidad_dudosa"].str.strip() == "True").astype(int)
df = df.drop(columns=["calidad_dudosa", "codigo"])

# ── Limpieza: eliminar valores centinela del sensor (ej. -999, -1000°C) ──────
# Se marcan como NaN los valores fuera del rango físico plausible
df = df.set_index("fecha_hora")
df["t"] = df["t"].where(
    (df["t"] >= cfg.temp_min) & (df["t"] <= cfg.temp_max)
)

# Interpolar huecos pequeños generados (hasta 10 min consecutivos)
df["t"] = df["t"].interpolate(method="time", limit=10)

df = df.reset_index()

# Eliminar filas con NaN restantes (huecos grandes o sin dato reparable)
df = df.dropna(subset=["t"]).reset_index(drop=True)

print(f"Filas totales       : {len(df):,}")
print(f"Rango               : {df['fecha_hora'].min()} → {df['fecha_hora'].max()}")
print(f"Anomalías           : {df['label'].sum():,} ({df['label'].mean()*100:.2f}%)")
print(f"Valores nulos en t  : {df['t'].isna().sum()}")
print(f"\nEstadísticos de temperatura (post limpieza):")
print(df["t"].describe().round(3))
assert df["t"].min() >= cfg.temp_min, "ERROR: quedan valores por debajo del límite físico"
assert df["t"].max() <= cfg.temp_max, "ERROR: quedan valores por encima del límite físico"
print(f"✓ Rango validado: [{df['t'].min():.2f}, {df['t'].max():.2f}] °C")

In [ ]:
# ── Celda 4: División cronológica (sin shuffle) ───────────────────────────────
n = len(df)
n_train = int(n * cfg.train_frac)
n_val   = int(n * cfg.val_frac)

train_df = df.iloc[:n_train].reset_index(drop=True)
val_df   = df.iloc[n_train : n_train + n_val].reset_index(drop=True)
test_df  = df.iloc[n_train + n_val :].reset_index(drop=True)

for name, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    a = split["label"].sum()
    print(f"{name:5s}: {len(split):6,} filas | anomalías: {a:4,} ({a/len(split)*100:.2f}%)")
    print(f"         {split['fecha_hora'].iloc[0]} → {split['fecha_hora'].iloc[-1]}")

In [ ]:
# ── Celda 5: Normalización MinMaxScaler — anti-sesgo ─────────────────────────
scaler = MinMaxScaler()
normal_train = train_df[train_df["label"] == 0][["t"]]
scaler.fit(normal_train)

train_vals  = scaler.transform(train_df[["t"]]).flatten()
val_vals    = scaler.transform(val_df[["t"]]).flatten()
test_vals   = scaler.transform(test_df[["t"]]).flatten()

train_labels = train_df["label"].values
val_labels   = val_df["label"].values
test_labels  = test_df["label"].values

print(f"Scaler min={scaler.data_min_[0]:.3f}, max={scaler.data_max_[0]:.3f}")
print(f"Rango train escalado: [{train_vals.min():.3f}, {train_vals.max():.3f}]")

In [ ]:
# ── Celda 6: Ventanas deslizantes ─────────────────────────────────────────────
def make_windows(arr, w=12):
    idx = np.arange(w)[None, :] + np.arange(len(arr) - w)[:, None]
    return arr[idx].astype(np.float32)

def make_labels_windows(labels, w=12):
    return np.array([labels[i:i+w].max() for i in range(len(labels) - w)], dtype=np.int64)

w = cfg.window_size
train_w  = make_windows(train_vals, w)
val_w    = make_windows(val_vals, w)
test_w   = make_windows(test_vals, w)

train_lw = make_labels_windows(train_labels, w)
val_lw   = make_labels_windows(val_labels, w)
test_lw  = make_labels_windows(test_labels, w)

for name, X, y in [("Train", train_w, train_lw), ("Val", val_w, val_lw), ("Test", test_w, test_lw)]:
    print(f"{name:5s}: ventanas={X.shape}, anomalías={y.sum()} ({y.mean()*100:.2f}%)")

In [ ]:
# ── Celda 7: DataLoaders ──────────────────────────────────────────────────────
train_tensor  = torch.FloatTensor(train_w)
val_tensor    = torch.FloatTensor(val_w)
test_tensor   = torch.FloatTensor(test_w)

train_ds_full = TensorDataset(train_tensor)

normal_idx = torch.where(torch.tensor(train_lw) == 0)[0]
train_loader = DataLoader(
    Subset(train_ds_full, normal_idx),
    batch_size=cfg.batch_size, shuffle=True, drop_last=True
)

val_loader  = DataLoader(TensorDataset(val_tensor),  batch_size=cfg.batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(test_tensor), batch_size=cfg.batch_size, shuffle=False)

print(f"Ventanas normales train : {len(normal_idx):,}")
print(f"Batches train           : {len(train_loader)}")
print(f"Batches val             : {len(val_loader)}")
print(f"Batches test            : {len(test_loader)}")

In [ ]:
# ── Celda 8: Modelo Transfer Learning (submatriz 1 sensor) ───────────────────
class SIATATransferModel(nn.Module):
    """
    Adapta USAD pre-entrenado (51 sensores, w=612) a 1 sensor (w=12).
    InputAdapter(12→306) + inner congelado + OutputAdapter1/2(306→12).
    """
    def __init__(self, checkpoint_path, w_size_new=12, z_size=120):
        super().__init__()
        ck = torch.load(checkpoint_path, map_location="cpu")

        self.input_adapter  = nn.Linear(w_size_new, 306)
        self.out_adapter1   = nn.Linear(306, w_size_new)
        self.out_adapter2   = nn.Linear(306, w_size_new)

        with torch.no_grad():
            self.input_adapter.weight.copy_(ck["encoder"]["linear1.weight"][:, :w_size_new])
            self.input_adapter.bias.copy_(ck["encoder"]["linear1.bias"])
            self.out_adapter1.weight.copy_(ck["decoder1"]["linear3.weight"][:w_size_new, :])
            self.out_adapter1.bias.copy_(ck["decoder1"]["linear3.bias"][:w_size_new])
            self.out_adapter2.weight.copy_(ck["decoder2"]["linear3.weight"][:w_size_new, :])
            self.out_adapter2.bias.copy_(ck["decoder2"]["linear3.bias"][:w_size_new])

        self.enc_l2 = nn.Linear(306, 153)
        self.enc_l3 = nn.Linear(153, z_size)
        with torch.no_grad():
            self.enc_l2.weight.copy_(ck["encoder"]["linear2.weight"])
            self.enc_l2.bias.copy_(ck["encoder"]["linear2.bias"])
            self.enc_l3.weight.copy_(ck["encoder"]["linear3.weight"])
            self.enc_l3.bias.copy_(ck["encoder"]["linear3.bias"])

        self.dec1_l1 = nn.Linear(z_size, 153)
        self.dec1_l2 = nn.Linear(153, 306)
        with torch.no_grad():
            self.dec1_l1.weight.copy_(ck["decoder1"]["linear1.weight"])
            self.dec1_l1.bias.copy_(ck["decoder1"]["linear1.bias"])
            self.dec1_l2.weight.copy_(ck["decoder1"]["linear2.weight"])
            self.dec1_l2.bias.copy_(ck["decoder1"]["linear2.bias"])

        self.dec2_l1 = nn.Linear(z_size, 153)
        self.dec2_l2 = nn.Linear(153, 306)
        with torch.no_grad():
            self.dec2_l1.weight.copy_(ck["decoder2"]["linear1.weight"])
            self.dec2_l1.bias.copy_(ck["decoder2"]["linear1.bias"])
            self.dec2_l2.weight.copy_(ck["decoder2"]["linear2.weight"])
            self.dec2_l2.bias.copy_(ck["decoder2"]["linear2.bias"])

        self._inner_params = (
            list(self.enc_l2.parameters()) + list(self.enc_l3.parameters()) +
            list(self.dec1_l1.parameters()) + list(self.dec1_l2.parameters()) +
            list(self.dec2_l1.parameters()) + list(self.dec2_l2.parameters())
        )
        self.freeze_inner()

    def freeze_inner(self):
        for p in self._inner_params:
            p.requires_grad = False

    def unfreeze_inner(self):
        for p in self._inner_params:
            p.requires_grad = True

    def _encode(self, x):
        h = F.relu(self.input_adapter(x))
        h = F.relu(self.enc_l2(h))
        return F.relu(self.enc_l3(h))

    def _decode1(self, z):
        h = F.relu(self.dec1_l1(z))
        h = F.relu(self.dec1_l2(h))
        return torch.sigmoid(self.out_adapter1(h))

    def _decode2(self, z):
        h = F.relu(self.dec2_l1(z))
        h = F.relu(self.dec2_l2(h))
        return torch.sigmoid(self.out_adapter2(h))

    def training_step(self, batch, n):
        z  = self._encode(batch)
        w1 = self._decode1(z)
        w2 = self._decode2(z)
        w3 = self._decode2(self._encode(w1))
        loss1 = (1/n)*F.mse_loss(w1,batch) + (1-1/n)*F.mse_loss(w3,batch)
        loss2 = (1/n)*F.mse_loss(w2,batch) - (1-1/n)*F.mse_loss(w3,batch)
        return loss1, loss2

    def validation_step(self, batch, n):
        with torch.no_grad():
            z  = self._encode(batch)
            w1 = self._decode1(z)
            w2 = self._decode2(z)
            w3 = self._decode2(self._encode(w1))
            loss1 = (1/n)*F.mse_loss(w1,batch) + (1-1/n)*F.mse_loss(w3,batch)
            loss2 = (1/n)*F.mse_loss(w2,batch) - (1-1/n)*F.mse_loss(w3,batch)
        return loss1.item(), loss2.item()

    @torch.no_grad()
    def score(self, loader, alpha=0.5, beta=0.5):
        scores = []
        for [batch] in loader:
            batch = to_device(batch, device)
            w1 = self._decode1(self._encode(batch))
            w2 = self._decode2(self._encode(w1))
            s  = alpha * torch.mean((batch-w1)**2, dim=1) + \
                 beta  * torch.mean((batch-w2)**2, dim=1)
            scores.append(s.cpu().numpy())
        return np.concatenate(scores)


model = SIATATransferModel(cfg.model_path, w_size_new=cfg.window_size, z_size=cfg.z_size)
model = to_device(model, device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros totales     : {total:,}")
print(f"Parámetros entrenables : {trainable:,} ({trainable/total*100:.1f}%)")

In [ ]:
# ── Celda 9: Entrenamiento 2 fases (con gradient clipping) ───────────────────
def make_optimizers(model, lr, wd):
    adapter_params = (
        list(model.input_adapter.parameters()) +
        list(model.out_adapter1.parameters()) +
        list(model.out_adapter2.parameters())
    )
    all_params1 = adapter_params + list(model.enc_l2.parameters()) + \
                  list(model.enc_l3.parameters()) + list(model.dec1_l1.parameters()) + \
                  list(model.dec1_l2.parameters())
    all_params2 = adapter_params + list(model.enc_l2.parameters()) + \
                  list(model.enc_l3.parameters()) + list(model.dec2_l1.parameters()) + \
                  list(model.dec2_l2.parameters())
    opt1 = torch.optim.Adam(all_params1, lr=lr, weight_decay=wd)
    opt2 = torch.optim.Adam(all_params2, lr=lr, weight_decay=wd)
    return opt1, opt2

history = {"train_l1": [], "train_l2": [], "val_l1": [], "val_l2": [], "phase": []}

# ── Fase 1: inner congelado ───────────────────────────────────────────────────
print("=== Fase 1: adapters (inner congelado) ===")
model.freeze_inner()
opt1, opt2 = make_optimizers(model, cfg.lr_frozen, cfg.weight_decay)
sched1 = torch.optim.lr_scheduler.ReduceLROnPlateau(opt1, mode='min', factor=0.5, patience=5)
sched2 = torch.optim.lr_scheduler.ReduceLROnPlateau(opt2, mode='min', factor=0.5, patience=5)

for epoch in range(1, cfg.epochs_frozen + 1):
    model.train()
    ep_l1, ep_l2 = [], []
    for [batch] in train_loader:
        batch = to_device(batch, device)
        l1, l2 = model.training_step(batch, epoch)
        l1.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt1.step(); opt1.zero_grad()
        l1, l2 = model.training_step(batch, epoch)
        l2.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt2.step(); opt2.zero_grad()
        ep_l1.append(l1.item()); ep_l2.append(l2.item())

    model.eval()
    vl1, vl2 = [], []
    for [batch] in val_loader:
        vl1_, vl2_ = model.validation_step(to_device(batch, device), epoch)
        vl1.append(vl1_); vl2.append(vl2_)

    m_l1, m_l2 = np.mean(ep_l1), np.mean(ep_l2)
    m_vl1 = np.mean(vl1)
    history["train_l1"].append(m_l1); history["train_l2"].append(m_l2)
    history["val_l1"].append(m_vl1);  history["val_l2"].append(np.mean(vl2))
    history["phase"].append(1)
    sched1.step(m_vl1); sched2.step(m_vl1)
    if epoch % 5 == 0:
        print(f"  Epoch {epoch:3d} | train_l1={m_l1:.6f} train_l2={m_l2:.6f} val_l1={m_vl1:.6f}")

# ── Fase 2: fine-tune completo ────────────────────────────────────────────────
print("\n=== Fase 2: fine-tune completo ===")
model.unfreeze_inner()
opt1, opt2 = make_optimizers(model, cfg.lr_finetune, cfg.weight_decay)
sched1 = torch.optim.lr_scheduler.ReduceLROnPlateau(opt1, mode='min', factor=0.5, patience=10)
sched2 = torch.optim.lr_scheduler.ReduceLROnPlateau(opt2, mode='min', factor=0.5, patience=10)

for epoch in range(cfg.epochs_frozen + 1, cfg.epochs_total + 1):
    model.train()
    ep_l1, ep_l2 = [], []
    for [batch] in train_loader:
        batch = to_device(batch, device)
        l1, l2 = model.training_step(batch, epoch)
        l1.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt1.step(); opt1.zero_grad()
        l1, l2 = model.training_step(batch, epoch)
        l2.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt2.step(); opt2.zero_grad()
        ep_l1.append(l1.item()); ep_l2.append(l2.item())

    model.eval()
    vl1, vl2 = [], []
    for [batch] in val_loader:
        vl1_, vl2_ = model.validation_step(to_device(batch, device), epoch)
        vl1.append(vl1_); vl2.append(vl2_)

    m_l1, m_l2 = np.mean(ep_l1), np.mean(ep_l2)
    m_vl1 = np.mean(vl1)
    history["train_l1"].append(m_l1); history["train_l2"].append(m_l2)
    history["val_l1"].append(m_vl1);  history["val_l2"].append(np.mean(vl2))
    history["phase"].append(2)
    sched1.step(m_vl1); sched2.step(m_vl1)
    if epoch % 10 == 0:
        print(f"  Epoch {epoch:3d} | train_l1={m_l1:.6f} train_l2={m_l2:.6f} val_l1={m_vl1:.6f}")

print("\nEntrenamiento completado.")

In [ ]:
# ── Celda 10: Puntuaciones de anomalía ────────────────────────────────────────
model.eval()

train_loader_full = DataLoader(train_ds_full, batch_size=cfg.batch_size, shuffle=False)
scores_train = model.score(train_loader_full, cfg.alpha, cfg.beta)
scores_val   = model.score(val_loader,   cfg.alpha, cfg.beta)
scores_test  = model.score(test_loader,  cfg.alpha, cfg.beta)

print(f"Scores train : min={scores_train.min():.6f}  max={scores_train.max():.6f}  mean={scores_train.mean():.6f}")
print(f"Scores val   : min={scores_val.min():.6f}  max={scores_val.max():.6f}  mean={scores_val.mean():.6f}")
print(f"Scores test  : min={scores_test.min():.6f}  max={scores_test.max():.6f}  mean={scores_test.mean():.6f}")

In [ ]:
# ── Celda 11: Selección de umbral con detección de score invertido ────────────

# ── Paso 1: ROC inicial para detectar inversión ───────────────────────────────
fpr_v, tpr_v, thresholds_v = roc_curve(val_lw, scores_val)
auc_val_raw = roc_auc_score(val_lw, scores_val)
print(f"AUC Val (raw)         : {auc_val_raw:.4f}")

# ── Paso 2: Corrección si AUC < 0.5 (score invertido) ───────────────────────
scores_flipped = False
if auc_val_raw < 0.5:
    print(f"⚠  AUC < 0.5 → score INVERSAMENTE correlacionado con anomalías.")
    print(f"   Causa probable: anomalías SIATA son fáciles de reconstruir (sensor")
    print(f"   congelado/drift), mientras datos normales con variabilidad alta")
    print(f"   producen mayor MSE. Aplicando transformación: score = max - score.")
    scores_flipped = True
    smax = max(scores_train.max(), scores_val.max(), scores_test.max())
    scores_train = smax - scores_train
    scores_val   = smax - scores_val
    scores_test  = smax - scores_test
    # Recalcular ROC con scores corregidos
    fpr_v, tpr_v, thresholds_v = roc_curve(val_lw, scores_val)
    auc_val = roc_auc_score(val_lw, scores_val)
    print(f"   AUC Val (corregido) : {auc_val:.4f}")
else:
    auc_val = auc_val_raw
    print(f"   AUC Val OK (≥ 0.5), no se requiere inversión.")

# ── Paso 3: Youden J como referencia ─────────────────────────────────────────
youden_idx = np.argmax(tpr_v - fpr_v)
threshold_youden = float(thresholds_v[youden_idx])
print(f"\nUmbral Youden J (referencia) : {threshold_youden:.8f}")
print(f"  TPR={tpr_v[youden_idx]:.4f}  FPR={fpr_v[youden_idx]:.4f}")

# ── Paso 4: F1-optimal sobre Precision-Recall (umbral principal) ─────────────
precision_pr, recall_pr, pr_thresholds = precision_recall_curve(val_lw, scores_val)
f1_pr = 2 * precision_pr[:-1] * recall_pr[:-1] / (precision_pr[:-1] + recall_pr[:-1] + 1e-8)
best_f1_idx  = np.argmax(f1_pr)
threshold    = float(pr_thresholds[best_f1_idx])
ap_val       = average_precision_score(val_lw, scores_val)

print(f"\nUmbral F1-optimal (principal): {threshold:.8f}")
print(f"  F1={f1_pr[best_f1_idx]:.4f}  Precision={precision_pr[best_f1_idx]:.4f}  Recall={recall_pr[best_f1_idx]:.4f}")
print(f"  Average Precision (AP)={ap_val:.4f}")

In [ ]:
# ── Celda 12: Métricas con Balanced Accuracy ──────────────────────────────────
pred_train = (scores_train >= threshold).astype(int)
pred_val   = (scores_val   >= threshold).astype(int)
pred_test  = (scores_test  >= threshold).astype(int)

def report(name, y_true, y_pred, y_score):
    ba  = balanced_accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    pre = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_score) if y_true.sum() > 0 else float('nan')
    print(f"[{name}]")
    print(f"  Balanced Accuracy : {ba:.4f}")
    print(f"  F1                : {f1:.4f}")
    print(f"  Precision         : {pre:.4f}")
    print(f"  Recall            : {rec:.4f}")
    print(f"  AUC-ROC           : {auc:.4f}")
    return dict(name=name, BA=ba, F1=f1, Precision=pre, Recall=rec, AUC=auc)

results = [
    report("Train", train_lw, pred_train, scores_train),
    report("Val",   val_lw,   pred_val,   scores_val),
    report("Test",  test_lw,  pred_test,  scores_test),
]
results_df = pd.DataFrame(results).set_index("name")
print("\n", results_df.round(4))

## Visualizaciones

In [ ]:
# ── Serie temporal completa con splits y anomalías ────────────────────────────
fig, ax = plt.subplots(figsize=(16, 4))

ax.plot(df["fecha_hora"], df["t"], color="steelblue", linewidth=0.5, label="Temperatura")

ax.axvspan(train_df["fecha_hora"].iloc[0],  train_df["fecha_hora"].iloc[-1],
           alpha=0.08, color="green",  label="Train (65%)")
ax.axvspan(val_df["fecha_hora"].iloc[0],    val_df["fecha_hora"].iloc[-1],
           alpha=0.12, color="orange", label="Val (20%)")
ax.axvspan(test_df["fecha_hora"].iloc[0],   test_df["fecha_hora"].iloc[-1],
           alpha=0.12, color="red",    label="Test (15%)")

anom = df[df["label"] == 1]
ax.scatter(anom["fecha_hora"], anom["t"], color="red", s=4, zorder=5, label="Anomalía real")

ax.set_title("Estación 203 (UNAN) — Temperatura 2023-01-01 a 2023-06-30", fontsize=13)
ax.set_xlabel("Fecha"); ax.set_ylabel("Temperatura (°C)")
ax.legend(loc="upper right", fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Curvas de entrenamiento ────────────────────────────────────────────────────
epochs_range = range(1, cfg.epochs_total + 1)
phase_break  = cfg.epochs_frozen

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(epochs_range, history["train_l1"], label="Train Loss1 (AE1)", color="steelblue")
ax.plot(epochs_range, history["train_l2"], label="Train Loss2 (AE2)", color="darkorange")
ax.plot(epochs_range, history["val_l1"],   label="Val Loss1",  color="green", linestyle="--")
ax.axvline(phase_break + 0.5, color="gray", linestyle=":", linewidth=1.5,
           label=f"Fase 2 (época {phase_break+1})")
ax.set_title("Curvas de Pérdida — Transfer Learning USAD", fontsize=13)
ax.set_xlabel("Época"); ax.set_ylabel("Loss (MSE)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Curva ROC — Validación ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr_v, tpr_v, color="steelblue", label=f"AUC = {auc_val:.4f}")
ax.plot([0,1],[0,1], "r:", label="Azar")
ax.scatter(fpr_v[youden_idx], tpr_v[youden_idx], color="red", zorder=5,
           label=f"Umbral Youden = {threshold_youden:.6f}")
ax.set_title("ROC — Validación" + (" (score corregido)" if scores_flipped else ""), fontsize=13)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Curva Precision-Recall — Validación (Plotly) ──────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=recall_pr, y=precision_pr, mode='lines',
    name=f'AP = {ap_val:.4f}',
    line=dict(color='steelblue', width=2)
))
fig.add_trace(go.Scatter(
    x=[recall_pr[best_f1_idx]],
    y=[precision_pr[best_f1_idx]],
    mode='markers',
    name=f'F1-opt = {f1_pr[best_f1_idx]:.4f} | umbral={threshold:.6f}',
    marker=dict(color='red', size=10, symbol='circle')
))
# Línea de referencia (baseline = tasa de anomalías)
baseline = val_lw.mean()
fig.add_hline(y=baseline, line_dash='dot', line_color='gray',
              annotation_text=f'Baseline ({baseline*100:.1f}%)',
              annotation_position='bottom right')
fig.update_layout(
    title='Curva Precision-Recall — Validación',
    xaxis_title='Recall',
    yaxis_title='Precision',
    template='plotly_white',
    height=450,
    legend=dict(x=0.01, y=0.01)
)
fig.show()

In [ ]:
# ── Distribución de scores por clase — Validación (Plotly) ───────────────────
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=scores_val[val_lw == 0],
    name='Normal',
    opacity=0.7,
    nbinsx=150,
    marker_color='steelblue'
))
fig.add_trace(go.Histogram(
    x=scores_val[val_lw == 1],
    name='Anomalía',
    opacity=0.7,
    nbinsx=150,
    marker_color='crimson'
))
fig.add_vline(
    x=threshold,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Umbral F1-opt={threshold:.6f}',
    annotation_position='top right'
)
fig.update_layout(
    barmode='overlay',
    title='Distribución de Scores USAD por Clase — Validación',
    xaxis_title='Score USAD' + (' (transformado: max-score)' if scores_flipped else ''),
    yaxis_title='Frecuencia',
    template='plotly_white',
    height=400
)
fig.show()

In [ ]:
# ── Error de Reconstrucción — Validación (Plotly interactivo) ─────────────────
def plot_recon_error_plotly(scores, labels, threshold, title, line_color='steelblue'):
    n = len(scores)
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        y=scores,
        x=list(range(n)),
        mode='lines',
        name='Score USAD',
        line=dict(color=line_color, width=0.8),
        hovertemplate='Ventana %{x}<br>Score: %{y:.6f}<extra></extra>'
    ))

    fig.add_hline(
        y=threshold,
        line_dash='dash',
        line_color='red',
        annotation_text=f'Umbral F1-opt={threshold:.6f}',
        annotation_position='bottom right'
    )

    # Regiones de anomalías reales sombreadas
    shapes = []
    in_anom, start = False, 0
    for i, lbl in enumerate(labels):
        if lbl == 1 and not in_anom:
            start, in_anom = i, True
        elif lbl == 0 and in_anom:
            shapes.append(dict(
                type='rect', x0=start, x1=i,
                y0=0, y1=1, xref='x', yref='paper',
                fillcolor='rgba(220,50,47,0.15)', line_width=0
            ))
            in_anom = False
    if in_anom:
        shapes.append(dict(
            type='rect', x0=start, x1=n-1,
            y0=0, y1=1, xref='x', yref='paper',
            fillcolor='rgba(220,50,47,0.15)', line_width=0
        ))

    fig.update_layout(
        shapes=shapes,
        title=title,
        xaxis_title='Ventana',
        yaxis_title='Score USAD' + (' (transformado)' if scores_flipped else ''),
        template='plotly_white',
        height=420,
        hovermode='x'
    )
    return fig

fig_val_err = plot_recon_error_plotly(
    scores_val, val_lw, threshold,
    'Error de Reconstrucción — Validación'
)
fig_val_err.show()

In [ ]:
# ── Error de Reconstrucción — Test (Plotly interactivo) ───────────────────────
fig_test_err = plot_recon_error_plotly(
    scores_test, test_lw, threshold,
    'Error de Reconstrucción — Test',
    line_color='darkorange'
)
fig_test_err.show()

In [ ]:
# ── Reconstrucción vs Original — Train (muestra 500 ventanas, matplotlib) ─────
@torch.no_grad()
def reconstruct(model, windows_np):
    x = to_device(torch.FloatTensor(windows_np), device)
    z  = model._encode(x)
    w1 = model._decode1(z)
    return w1.cpu().numpy()

sample_idx = slice(0, 500)
orig_train  = train_w[sample_idx]
recon_train = reconstruct(model, orig_train)

orig_series  = orig_train[:, -1]
recon_series = recon_train[:, -1]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(orig_series,  color="steelblue",  linewidth=0.8, label="Original")
ax.plot(recon_series, color="darkorange", linewidth=0.8, linestyle="--", label="Reconstruida")
ax.set_title("Reconstrucción — Train (primeras 500 ventanas)", fontsize=13)
ax.set_xlabel("Ventana"); ax.set_ylabel("Temperatura (escalada)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Reconstrucción vs Original — Validación (matplotlib) ─────────────────────
recon_val = reconstruct(model, val_w)
orig_series_v  = val_w[:, -1]
recon_series_v = recon_val[:, -1]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(orig_series_v,  color="steelblue",  linewidth=0.6, label="Original")
ax.plot(recon_series_v, color="darkorange", linewidth=0.6, linestyle="--", label="Reconstruida")

det_idx  = np.where(pred_val == 1)[0]
real_idx = np.where(val_lw  == 1)[0]
ax.scatter(det_idx,  orig_series_v[det_idx],  color="purple", marker="x", s=20,
           zorder=5, label="Detectada")
ax.scatter(real_idx, orig_series_v[real_idx], color="red",    s=6,
           zorder=4, label="Real")

ax.set_title("Reconstrucción — Validación", fontsize=13)
ax.set_xlabel("Ventana"); ax.set_ylabel("Temperatura (escalada)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Reconstrucción vs Original — Test (matplotlib) ────────────────────────────
recon_test = reconstruct(model, test_w)
orig_series_t  = test_w[:, -1]
recon_series_t = recon_test[:, -1]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(orig_series_t,  color="steelblue",  linewidth=0.6, label="Original")
ax.plot(recon_series_t, color="darkorange", linewidth=0.6, linestyle="--", label="Reconstruida")

det_idx_t  = np.where(pred_test == 1)[0]
real_idx_t = np.where(test_lw  == 1)[0]
ax.scatter(det_idx_t,  orig_series_t[det_idx_t],  color="purple", marker="x", s=20,
           zorder=5, label="Detectada")
ax.scatter(real_idx_t, orig_series_t[real_idx_t], color="red",    s=6,
           zorder=4, label="Real")

ax.set_title("Reconstrucción — Test", fontsize=13)
ax.set_xlabel("Ventana"); ax.set_ylabel("Temperatura (escalada)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Matrices de Confusión ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, y_true, y_pred) in zip(
    axes,
    [("Train", train_lw, pred_train),
     ("Val",   val_lw,   pred_val),
     ("Test",  test_lw,  pred_test)]
):
    cm = confusion_matrix(y_true, y_pred)
    ba = balanced_accuracy_score(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Normal","Anomalía"],
                yticklabels=["Normal","Anomalía"])
    ax.set_title(f"{name}\nBalanced Acc = {ba:.4f}", fontsize=11)
    ax.set_xlabel("Predicho"); ax.set_ylabel("Real")

plt.suptitle("Matrices de Confusión — USAD Transfer Learning Sensor 203 v.K",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("\n── Resumen de métricas ──")
print(results_df.round(4).to_string())

In [ ]:
# ── Reconstrucción vs Original + Anomalías con eje de fechas (Plotly) ─────────
# Inversa del scaler para pasar de escala [0,1] a °C

def plot_reconstruction_dates_plotly(df_split, recon_arr, pred_lw, real_lw,
                                     scaler, window_size, title,
                                     max_display=60_000):
    """
    Gráfica interactiva con eje de fechas en °C.
    Alinea cada ventana con la fecha del último punto (w-1 offset).
    """
    n_windows = len(recon_arr)
    # Última columna de cada ventana → reconstrucción en escala
    recon_scaled = recon_arr[:, -1].reshape(-1, 1)
    recon_real   = scaler.inverse_transform(recon_scaled).flatten()

    # Fechas del punto final de cada ventana
    offset = window_size - 1
    dates    = df_split["fecha_hora"].iloc[offset : offset + n_windows].values
    orig_real = df_split["t"].iloc[offset : offset + n_windows].values

    # Submuestreo para rendimiento si hay demasiados puntos
    if n_windows > max_display:
        step = n_windows // max_display
        dates      = dates[::step]
        orig_real  = orig_real[::step]
        recon_real = recon_real[::step]
        pred_idx_mask = np.arange(0, n_windows, step)
        real_idx_mask = np.arange(0, n_windows, step)
    else:
        pred_idx_mask = np.arange(n_windows)
        real_idx_mask = np.arange(n_windows)

    fig = go.Figure()

    # Original
    fig.add_trace(go.Scatter(
        x=dates, y=orig_real, mode='lines', name='Original',
        line=dict(color='steelblue', width=0.9),
        hovertemplate='%{x}<br>Orig: %{y:.2f} °C<extra></extra>'
    ))

    # Reconstruida
    fig.add_trace(go.Scatter(
        x=dates, y=recon_real, mode='lines', name='Reconstruida',
        line=dict(color='darkorange', width=0.9, dash='dot'),
        hovertemplate='%{x}<br>Recon: %{y:.2f} °C<extra></extra>'
    ))

    # Anomalías reales
    real_idx = np.where(real_lw[:n_windows] == 1)[0]
    if len(real_idx) > 0:
        rdates = df_split["fecha_hora"].iloc[offset + real_idx].values
        rtemps = df_split["t"].iloc[offset + real_idx].values
        fig.add_trace(go.Scatter(
            x=rdates, y=rtemps, mode='markers', name='Anomalías reales',
            marker=dict(color='red', size=4, opacity=0.7),
            hovertemplate='%{x}<br>%{y:.2f} °C<extra>Anomalía real</extra>'
        ))

    # Anomalías predichas
    pred_idx = np.where(pred_lw[:n_windows] == 1)[0]
    if len(pred_idx) > 0:
        pdates = df_split["fecha_hora"].iloc[offset + pred_idx].values
        ptemps = df_split["t"].iloc[offset + pred_idx].values
        fig.add_trace(go.Scatter(
            x=pdates, y=ptemps, mode='markers', name='Anomalías predichas',
            marker=dict(color='purple', size=5, symbol='x', opacity=0.8),
            hovertemplate='%{x}<br>%{y:.2f} °C<extra>Predicha</extra>'
        ))

    fig.update_layout(
        title=title,
        xaxis_title='Fecha',
        yaxis_title='Temperatura (°C)',
        template='plotly_white',
        height=470,
        hovermode='x unified',
        legend=dict(x=0.01, y=0.99)
    )
    return fig

# Validación
fig_val_dates = plot_reconstruction_dates_plotly(
    val_df, recon_val, pred_val, val_lw,
    scaler, cfg.window_size,
    'Reconstrucción vs Original + Anomalías — USAD TL (Validación)'
)
fig_val_dates.show()

# Test
fig_test_dates = plot_reconstruction_dates_plotly(
    test_df, recon_test, pred_test, test_lw,
    scaler, cfg.window_size,
    'Reconstrucción vs Original + Anomalías — USAD TL (Test)'
)
fig_test_dates.show()

In [ ]:
# ── Celda final: Guardar modelo y artefactos ──────────────────────────────────
save_path = "modelos/usad/model_siata_203_k.pth"
torch.save({
    "model_state": model.state_dict(),
    "threshold": threshold,
    "threshold_youden": threshold_youden,
    "scores_flipped": scores_flipped,
    "scaler_min": float(scaler.data_min_[0]),
    "scaler_max": float(scaler.data_max_[0]),
    "config": {
        "station_code": cfg.station_code,
        "fecha_inicio": cfg.fecha_inicio,
        "fecha_fin": cfg.fecha_fin,
        "window_size": cfg.window_size,
        "z_size": cfg.z_size,
        "alpha": cfg.alpha,
        "beta": cfg.beta,
        "temp_min": cfg.temp_min,
        "temp_max": cfg.temp_max,
    }
}, save_path)
print(f"Modelo guardado en  : {save_path}")
print(f"Score invertido     : {scores_flipped}")
print(f"Umbral F1-optimal   : {threshold:.8f}")
print(f"Umbral Youden J     : {threshold_youden:.8f} (referencia)")
print(f"Balanced Acc Test   : {balanced_accuracy_score(test_lw, pred_test):.4f}")